from pyspark.sql.functions import *

In [ ]:
%sql

CREATE OR REPLACE TABLE lakehouse_queimada.bronze.queimadas AS
SELECT *
FROM READ_FILES('/Volumes/lakehouse_queimada/bronze/raw',
    FORMAT => 'CSV', 
    HEADER => TRUE, 
    INFERSCHEMA => TRUE, 
    SCHEMAEVOLUTIONMODE => 'NONE'
    )

In [ ]:
#Conectando com a base de dados

database_name = 'lakehouse_queimada'
medalhao_name = 'bronze'
table_name = 'queimadas'

df = spark.read.table(f'{database_name}.{medalhao_name}.{table_name}')

In [ ]:
# Ajustando o campo de nulo
# Valores como 'Dias_sem_chuva" = -999 significa que esta "NULO" sem informação

df = df.withColumn(
    'numero_dias_sem_chuva',
    when(
        col('numero_dias_sem_chuva') == -999, None
        )
    .otherwise(col('numero_dias_sem_chuva'))
    )
df = df.withColumn(
    'precipitacao',
    concat((round(col('precipitacao') * 100, 0)).cast('int').cast('string'), lit('%'))
)
df = df.withColumn(
    'risco_fogo',
    concat((round(col('risco_fogo') * 100, 0)).cast('int').cast('string'), lit('%'))
)

In [ ]:
df_colunas = df[
    "id",
    "lat", 
    "lon", 
    "data_hora_gmt", 
    "satelite", 
    "municipio", 
    "estado", 
    "pais", 
    "numero_dias_sem_chuva", 
    "precipitacao", 
    "risco_fogo", 
    "bioma", 
    "frp"
    ]
display(df_colunas)

In [ ]:
database_name = 'lakehouse_queimada'
medalhao_name = 'gold'
table_name = 'queimadas'

df_colunas.write \
    .mode('overwrite') \
    .saveAsTable(f'{database_name}.{medalhao_name}.{table_name}')

print(f'Tabela {database_name}.{medalhao_name}.{table_name} salva com sucesso!')